# Step2: Annotation and Malignancy

Evidence-first analysis notebook for clustering review, marker/CellTypist/LLM annotation evidence, consensus labels, and optional malignancy interpretation.

Input: `data/processed/Step2-sce_preprocessed.h5ad`  
Output: `data/processed/Step3-sce_annotated.h5ad` plus reviewable artifacts under `results/analysis_acceptance/`.

This notebook intentionally calls `scripts/run_analysis_acceptance.py` so the interactive workflow and command-line acceptance runner stay synchronized.

---

## 1. Setup

In [ ]:
from pathlib import Path
import importlib.util
import json

import pandas as pd
import scanpy as sc

import scLucid as scl

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "run_analysis_acceptance.py").exists():
    PROJECT_ROOT = Path("/Users/luye/Scripts/scLucid")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "analysis_acceptance"
INPUT_H5AD = DATA_DIR / "Step2-sce_preprocessed.h5ad"
OUTPUT_H5AD = RESULTS_DIR / "Step3-sce_annotated.h5ad"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_H5AD}")
print(f"Output dir: {RESULTS_DIR}")

## 2. Configure The Acceptance Run

Default first-pass annotation is conservative and evidence-first. CellTypist is optional; LLM output is treated as evidence only when you provide precomputed labels or a callback in code. Malignancy interpretation is a lightweight bridge that consumes annotation/tumor marker/CNV/signature evidence; heavy tumor algorithms remain in `scLucid.tumor`.

In [ ]:
RUN_CELLTYPIST = False
RUN_MALIGNANCY = True
RUN_CNV_FOR_MALIGNANCY = False

SPECIES = "human"
TISSUE = None          # e.g. "Pancreas", "Lung"
CANCER_TYPE = None    # e.g. "PDAC", "Lung Cancer"

RESOLUTIONS = (0.5, 0.6, 0.8)
N_CELLS = None        # set to an integer for a quick smoke run
RANDOM_STATE = 42

MARKER_CONFIG = None  # optional custom TOML marker file
CELLTYPIST_MODEL = "Immune_All_Low.pkl"

## 3. Run Analysis Acceptance

In [ ]:
runner_path = PROJECT_ROOT / "scripts" / "run_analysis_acceptance.py"
spec = importlib.util.spec_from_file_location("run_analysis_acceptance", runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

manifest = runner.run_analysis_acceptance(
    input_path=INPUT_H5AD,
    output_dir=RESULTS_DIR,
    n_cells=N_CELLS,
    resolutions=RESOLUTIONS,
    marker_config=MARKER_CONFIG,
    species=SPECIES,
    tissue=TISSUE,
    run_celltypist=RUN_CELLTYPIST,
    celltypist_model=CELLTYPIST_MODEL,
    run_malignancy=RUN_MALIGNANCY,
    run_cnv=RUN_CNV_FOR_MALIGNANCY,
    cancer_type=CANCER_TYPE,
    random_state=RANDOM_STATE,
    overwrite=True,
    show_progress=True,
    write_h5ad=True,
)

print(json.dumps(manifest["acceptance"], indent=2, ensure_ascii=False))

## 4. Inspect Acceptance Status

In [ ]:
acceptance = manifest["acceptance"]
metrics = pd.Series(acceptance["metrics"], name="value").to_frame()
display(metrics)

if acceptance["blocking_failures"]:
    print("Blocking failures:")
    for item in acceptance["blocking_failures"]:
        print(f"- {item}")

if acceptance["warnings"]:
    print("Warnings / review reasons:")
    for item in acceptance["warnings"]:
        print(f"- {item}")

## 5. Review Clustering And Annotation Evidence

In [ ]:
adata = sc.read_h5ad(manifest["artifacts"]["final_h5ad"])
analysis_dir = Path(manifest["artifacts"]["analysis_dir"])

review_table_path = analysis_dir / "annotation_review_table.csv"
annotation_review = pd.read_csv(review_table_path)
display(annotation_review)

compact_summary_path = analysis_dir / "analysis_review_summary_compact.json"
compact_summary = json.loads(compact_summary_path.read_text())
print(json.dumps(compact_summary, indent=2, ensure_ascii=False))

## 6. Inspect LLM Annotation Bundle

The bundle is the recommended payload to send to an LLM for data-driven annotation. It contains cluster sizes, informative markers, noisy markers, marker-manager evidence, and optional reference summaries. The LLM response should come back as cluster-level evidence, not direct truth.

In [ ]:
llm_bundle_path = analysis_dir / "llm_annotation_bundle.json"
llm_bundle = json.loads(llm_bundle_path.read_text())

print(llm_bundle["instructions"])
first_cluster = next(iter(llm_bundle["clusters"]))
print(f"
Example cluster: {first_cluster}")
print(json.dumps(llm_bundle["clusters"][first_cluster], indent=2, ensure_ascii=False)[:3000])

## 7. Inspect Malignancy Interpretation

In [ ]:
malignancy_table_path = analysis_dir / "malignancy_interpretation_table.csv"
if malignancy_table_path.exists():
    malignancy_review = pd.read_csv(malignancy_table_path)
    display(malignancy_review)
else:
    print("Malignancy interpretation was not enabled for this run.")

for key in ["malignancy_call", "malignancy_interpretation_score", "malignancy_call_basis"]:
    if key in adata.obs:
        print(f"{key}:")
        display(adata.obs[key].value_counts(dropna=False).to_frame())

## 8. Quick Visualization

In [ ]:
if "X_umap" not in adata.obsm:
    sc.tl.umap(adata)

plot_keys = [key for key in [
    "leiden_clusters",
    "cell_type_auto",
    "cell_type_auto_status",
    "malignancy_call",
    "malignancy_interpretation_score",
] if key in adata.obs]

sc.pl.umap(adata, color=plot_keys, ncols=2, frameon=False)

## 9. Save Project-Compatible Output

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
final_project_path = DATA_DIR / "Step3-sce_annotated.h5ad"
adata.write(final_project_path, compression="gzip")
print(f"Saved annotated object: {final_project_path}")
print(f"Acceptance manifest: {RESULTS_DIR / 'manifest.json'}")
print(f"Analysis artifacts: {RESULTS_DIR / 'analysis'}")

## 10. Human Review Checklist

Before using this object for downstream biology, inspect:

- whether the selected resolution gives biologically interpretable clusters;
- clusters with `needs_review = True` in `annotation_review_table.csv`;
- clusters dominated by ribosomal/mitochondrial/stress/hemoglobin markers;
- disagreement between marker-manager evidence, CellTypist/reference evidence, and any LLM annotation;
- whether malignancy calls rely on multiple evidence streams rather than epithelial identity alone.